# Experimentos, cambios en el pipeline y conclusiones


## 1. Preguntas de investigación y experimentos


### 1.1. Dataset


Ejecuta ambos modelos en Framingham, Telco y Credit con tres particiones y todo el entrenamiento:
3 datasets × 3 semillas × 2 modelos = 18 ejecuciones.
Uso los mismos parámetros en los tres modelos


<table>
<tr>
<td><img src="./graficas/datasets_seconds.png" alt="Tiempos por dataset" width="100%"></td>
<td><img src="./graficas/datasets_f1.png" alt="F1 por dataset" width="100%"></td>
<td><img src="./graficas/datasets_models.png" alt="F1 por dataset" width="100%"></td>
</tr>
</table>

El datasets de crédito es el que mas tarda debido a que es el que tiene mas datos. Podemos ver quelos dos modelos obtienen resultados similares.

### 1.2. Tamaño de muestra y semilla de partición

Usamos una batería de experimentos con 4 tamaños × 5 semillas × 2 modelos = 40 ejecuciones. Los tamaños son el 25 %, 50 % , 75% y 100% del entrenamiento disponible. En este caso añadimos como parámetros al lightgbm scale_pos_weight: 85 / 15 y al tbpfn balance_probabilities para mejorar los rendimientos debido al gran desbalanceo de mi dataset.


<table>
<tr>
<td><img src="./graficas/seed_partitions.png" alt="Tiempos por dataset" width="100%"></td>
<td><img src="./graficas/seed_split.png" alt="F1 por dataset" width="100%"></td>
</tr>
</table>

La semilla de partición no importa mucho.

El tamaño de la partición de entrenamiento si que importa y cuanto mayor sea mejor resultados se obtienen, como era de esperar.


### 1.3. Preprocesamiento


Se comparan tres variantes, dos modelos y cinco semillas: **30 ejecuciones**.

| Variante | Imputación numérica | Codificación categórica | Comparación |
|---|---|---|---|
| `minimo` | Mediana | Ordinal | Referencia mínima común |
| `one_hot` | Mediana | One-hot | efecto de la codificación |
| `media` | Media | Ordinal | efecto de la imputación |

<table>
<tr>
<td><img src="./graficas/pre.png" alt="Tiempos por dataset" width="100%"></td>
</tr>
</table>

No influye mucho ...


### 1.4. Tipo de variables


Probamos a entrenar solo con variables categórias, solo con númericas o con todas. En total hacemos 10 experimentos por cada una.

<table>
<tr>
<td><img src="./graficas/variables.png" alt="Tiempos por dataset" width="100%"></td>
<td><img src="./graficas/variables_2.png" alt="Tiempos por dataset" width="100%"></td>
</tr>
</table>

Las variables númericas son mas importantes que las categóricas y usarlas todas es mejor que un tipo solo.

Lightgbm sigue ofreciendo mejores resultados.

### 1.5. Hiperparámetros


En el caso de lightgbm ptimizamos hiperparametros y umbral. Algunos de los mas relevantes son los de abajo donde se puede ver la influencia en la f1.

<table>
<tr>
<td><img src="./graficas/nose.png" alt="Tiempos por dataset" width="100%"></td>
<td><img src="./graficas/model_lr.png" alt="F1 por dataset" width="100%"></td>
</tr>
</table>
<table>
<tr>
<td><img src="./graficas/model_nl.png" alt="Tiempos por dataset" width="100%"></td>
<td><img src="./graficas/model_nestimators.png" alt="F1 por dataset" width="100%"></td>
</tr>
</table>

En el caso de tabpfn ajustar el umbral y los hiperparámetros incrementa mucho la f1. También se puede ver la influencia de otros parámetros.

<table>
<tr>
<td><img src="./graficas/t1.png" alt="Tiempos por dataset" width="100%"></td>
<td><img src="./graficas/idk.png" alt="F1 por dataset" width="100%"></td>
</tr>
</table>
<table>
<tr>
<td><img src="./graficas/t2.png" alt="F1 por dataset" width="100%"></td>
<td><img src="./graficas/t3.png" alt="F1 por dataset" width="100%"></td>
</tr>
</table>

### 1.6. Coste


Esta es la diferencia para lightgbm en los tiempos entre ajustar hiperparametros o no y diferencia entre un lr bajo y uno alto.

<table>
<tr>
<td><img src="./graficas/sec.png" alt="Tiempos por dataset" width="100%"></td>
<td><img src="./graficas/sec_lr.png" alt="F1 por dataset" width="100%"></td>
</tr>
</table>

Para tabpfn ajustar hiperparametros es muy costoso. Ajustar el umbral no cuesta mucho y supone un incremento en la f1 notable por lo que compensa bastante hacerlo.

<table>
<tr>
<td><img src="./graficas/t4.png" alt="Tiempos por dataset" width="100%"></td>
</tr>
</table>

### 1.7. Calidad y ruido


Se comparan cinco variantes, tres semillas y dos modelos: **30 ejecuciones**. Las perturbaciones se aplican únicamente al entrenamiento; el test conserva los datos originales, incluidos sus valores ausentes naturales.

| Variante | Qué hace | Qué compara |
|---|---|---|
| `referencia` | Conserva el entrenamiento original | Referencia de esta batería |
| `ausentes_10` | Oculta cada celda observada con probabilidad 0.10 | Caída respecto a referencia |
| `ausentes_30` | Oculta cada celda observada con probabilidad 0.30 | Efecto de aumentar los ausentes |
| `ruido_20` | Añade ruido gaussiano a sysBP, diaBP, BMI, totChol y glucose | Sensibilidad al ruido de medida simulado |
| `menos_positivos` | Conserva la mitad de los positivos del entrenamiento | Efecto de una menor representación de la clase positiva |

<table>
<tr>
<td><img src="./graficas/noise1.png" alt="Tiempos por dataset" width="100%"></td>
<td><img src="./graficas/noise2.png" alt="F1 por dataset" width="100%"></td>
</tr>
</table>

En general los modelos son bastante robustos y toleran la presencia de ausentes y la de ruido. Al ser un caso muy desbalanceado la disminución de la clase positiva afecta mucho al modelo como era de esperar.

## 2. Cambios implementados en el pipeline


ARCHIVOS NUEVOS:
- batteries.py ejecuta los YAML con --config y registra plan, progreso y resultados en un padre de MLflow.parada temprana.
- partitions.py conserva particiones compartidas y tamaños anidados. 
- conditions.py permite los grupos de variables y perturbaciones necesarios.

MODIFICACIONES ARCHIVOS EXISTENTES:
- pipeline.py admite ejecuciones hijas y conserva tiempos separados, predicciones y condiciones. tracking.py expone tamaños y dimensiones como parámetros consultables en MLflow.
- optimization.py realiza el ajuste acotado dentro del entrenamiento. 
- evaluation.py registra AP además del resto de métricas.

Las baterías de experimentos se almacenan en configs/batteries/ 


## 3. Conclusiones

No merece la pena el uso de tabpfn por los siguientes motivos:
- Tarda mucho mas en ejecutarse, sobre todo en casos de optimización de hiperparámetros.
- Dependes de un servicio externo donde puedes alcanzar el límite de uso y tener que esperar al día siguiente.
- Los resultados no son mejores que con lightgbm.

La optimización de umbral en tabpfn merece mucho la pena, es indispensable.

Preguntas:
- Lo normal es analizar los resultados de las baterias de experimentos desde la app web de mlflow o desde código?
- 
